# Day 4 — Tensor Shapes Deep Dive

## 1. Learning Objectives
- Understand standard shape conventions used in deep learning.
- Visualize `[batch, features]` for tabular data.
- Visualize `[batch, channels, height, width]` for image data.
- Visualize `[sequence, embedding]` for text/time-series data.
- Debug shape mismatches conceptually.

## 3. Concept Explanation
In PyTorch, the **shape** of a tensor tells you exactly what kind of data it represents. By convention, the **first dimension (dim=0)** is almost always the **batch size**.

We don't feed one image into a neural network at a time. We feed a batch (e.g., 32 images) at once for parallel processing on the GPU. PyTorch needs a specific shape structure depending on the domain (tabular, vision, NLP).

## 4. Why This Matters
If you feed an image of shape `[3, 224, 224]` (Channels, Height, Width) into a model that expects `[1, 3, 224, 224]` (Batch, Channels, Height, Width), the code will crash. Shape conventions are the "language" different PyTorch layers use to talk to each other.

In [ ]:
import torch

## 5. Shape Convention 1: Tabular Data (Linear Layers)
**Shape:** `[batch_size, features]`

Imagine an Excel spreadsheet. Each row is a person, and each column is a feature (age, salary, height).
- If you process 32 people at once, and each person has 10 features, your tensor shape is `[32, 10]`.

In [ ]:
# 32 people, 10 features each
tabular_batch = torch.randn(32, 10)
print("Tabular Data Shape:", tabular_batch.shape)

## 6. Shape Convention 2: Computer Vision (CNNs)
**Shape:** `[batch_size, channels, height, width]` often written as `[N, C, H, W]`

- `N` = Batch Size (e.g., 32 images).
- `C` = Channels (3 for RGB, 1 for Grayscale).
- `H` = Height in pixels.
- `W` = Width in pixels.

*Note: Some other frameworks (like TensorFlow) use [N, H, W, C], but PyTorch strictly uses channels-first [N, C, H, W].*

In [ ]:
# 32 RGB images, each 224x224 pixels
image_batch = torch.randn(32, 3, 224, 224)
print("Image Batch Shape:", image_batch.shape)

## 7. Shape Convention 3: NLP and Time-Series (RNNs/Transformers)
**Shape:** `[batch_size, sequence_length, embedding_dim]` (Usually)

- `batch_size` = Number of sentences processed at once.
- `sequence_length` = Number of words (or tokens) in the sentence.
- `embedding_dim` = The size of the vector representing each word.

If we have 32 sentences, each 15 words long, and each word is represented by a 256-number vector, the shape is `[32, 15, 256]`.

In [ ]:
# 32 sentences, 15 words, 256 embedding dimension
nlp_batch = torch.randn(32, 15, 256)
print("NLP Batch Shape:", nlp_batch.shape)

## 11. Practice Exercise 1: Reshaping an Image into Features
A standard fully connected layer (like for tabular data) cannot accept a 4D image tensor `[N, C, H, W]`. It expects `[batch_size, features]`. 
To pass an image to a linear layer, you must **flatten** all dimensions *except* the batch dimension.

**Task:** Create a dummy image batch of shape `[16, 3, 64, 64]`. Flatten it into shape `[16, features]`. What is the value of `features`?

In [ ]:
# Write your code here

In [ ]:
# SOLUTION
images = torch.randn(16, 3, 64, 64)

# features = 3 * 64 * 64 = 12288
flattened = images.reshape(16, -1) # -1 tells PyTorch to figure out the remaining dimension
print("Flattened shape:", flattened.shape)

## 13. Debugging Challenge
You load a single RGB image of size 128x128 from disk. Its shape is `[3, 128, 128]`. You pass it to your Convolutional Neural Network, but the network crashes because it expects `[N, C, H, W]`.

How do you fix the shape of this single image before passing it to the model?

In [ ]:
single_image = torch.randn(3, 128, 128)
# CNN expects 4 dimensions. Fix it here:

**Solution:** You must add a dummy batch dimension of size 1 using `unsqueeze`.

In [ ]:
# Correct approach:
batched_image = single_image.unsqueeze(0)
print("Corrected shape:", batched_image.shape) # [1, 3, 128, 128]

## 17. Interview Questions
1. **What is the standard PyTorch format for an image tensor batch?**
   *Answer*: `[Batch, Channels, Height, Width]` (NCHW).
2. **If you have a tensor of shape `[32, 3, 224, 224]`, how many total pixels are in this batch (excluding channels)?**
   *Answer*: `32 * 224 * 224`. Each image is `224x224`, and there are 32 of them.

## 19. Day Summary
- Know the conventions: `[N, Features]` for linear, `[N, C, H, W]` for images, `[N, SeqLen, Embedding]` for text.
- Use `.unsqueeze(0)` to add a batch dimension to a single data point.
- Use `.reshape(batch_size, -1)` to flatten spatial dimensions into a 1D feature vector for linear layers.